# MVP v0.2.3: Negative Behavior Guidance

**Date:** 2026-03-12  
**Builds on:** MVP v0.2.2 (diffusion score guidance — positive only)

## What's new

v0.2 and v0.2.2 only used **positive guidance** — pushing actions toward the target policy π.
But SOPE's key contribution is **negative behavior guidance**: subtracting the behavior policy's
score to push *away* from the behavior distribution. The full guidance is:

```
guide = ∇_a log π(a|s) − ratio · ∇_a log β(a|s)
```

Without the negative term, the chunk diffuser's strong prior toward expert behavior dominates,
and guidance either has no effect (v0.2.2) or just degrades trajectories (v0.2).

## Changes from v0.2.2

1. **BC_Gaussian behavior policy β** — fitted to expert demo (state, action) pairs (full 19+7 dims)
2. **Negative guidance** — `guide = target_grad - ratio * behavior_grad` matching SOPE's `default_sample_fn`
3. **k_guide iterations** — multiple guidance steps per denoising step (SOPE default: 2)
4. **Adaptive cosine scaling** — guidance scale decays over diffusion timesteps
5. **Hyperparameter sweep** — over `action_scale` and `ratio`

## Pipeline

1. Load 200 expert demos (full 26-dim)
2. Load oracle values (pre-collected)
3. Load pre-trained chunk diffusion model (from v0.2.2)
4. Train BC_Gaussian behavior policy on expert demos
5. Load target policy + create diffusion scorer
6. Guided stitching with full SOPE guidance (positive + negative)
7. Evaluate OPE vs oracle

In [1]:
import sys, os
import importlib
import numpy as np
import torch
import torch.nn as nn
import h5py
import json
import math
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from collections import OrderedDict
from copy import deepcopy
from tqdm import tqdm
import time

# Project root
PROJECT_ROOT = Path("/home1/reishuen/latent_sope")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "sope"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "robomimic"))

# SOPE imports
from opelab.core.baselines.diffusion.temporal import TemporalUnet
from opelab.core.baselines.diffusion.diffusion import GaussianDiffusion
from opelab.core.baselines.diffusion.helpers import EMA, apply_conditioning

# Robomimic imports
import robomimic.utils.file_utils as FileUtils
import robomimic.utils.obs_utils as ObsUtils

# Our imports
import latent_sope.robomimic_interface.guidance as _guidance_mod
importlib.reload(_guidance_mod)
from latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint, build_algo_from_checkpoint
)
from latent_sope.robomimic_interface.guidance import RobomimicDiffusionScorer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Paths
DEMO_HDF5 = PROJECT_ROOT / "third_party/robomimic/datasets/lift/ph/low_dim_v15.hdf5"
CKPT_DIR = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models/test/20260309132349"
RESULTS_DIR = PROJECT_ROOT / "results/2026-03-12"
DIFFUSION_SAVE_DIR = PROJECT_ROOT / "diffusion_ckpts" / "mvp_v022_fulldim"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...23}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...23}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.l

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Device: cuda


## Configuration

In [2]:
# ── Obs keys (sorted, matching robomimic convention & LowDimConcatEncoder layout) ──
OBS_KEYS = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]

# Full dims — NO reduction (matching v0.2.2)
STATE_DIM = 19   # object(10) + eef_pos(3) + eef_quat(4) + gripper_qpos(2)
ACTION_DIM = 7   # full robomimic action space
TRANSITION_DIM = STATE_DIM + ACTION_DIM  # 26

# ── Diffusion config (must match v0.2.2 checkpoint) ──
CHUNK_SIZE = 4
N_DIFFUSION_STEPS = 256
DIM_MULTS = (1, 4, 8)
BASE_DIM = 32
ACTION_WEIGHT = 5.0
PREDICT_EPSILON = False  # x0-prediction (per D2.1 finding)

# ── Oracle config ──
ORACLE_JSON = CKPT_DIR / "oracle_50.json"

# ── OPE config ──
NUM_SYNTHETIC_TRAJS = 50
T_GEN = 60
GAMMA = 1.0

# ── Reward ──
CUBE_Z_INDEX = 2  # index 2 in the 19-dim latent (object key, cube_pos z)
LIFT_THRESHOLD = 0.84

# ── Behavior policy (diffusion-based) config ──
BEHAVIOR_HIDDEN_DIM = 256
BEHAVIOR_DIFFUSION_STEPS = 100
BEHAVIOR_TRAIN_EPOCHS = 1000
BC_LR = 1e-3
BC_BATCH_SIZE = 256

# ── Guidance config (full SOPE-style) ──
K_GUIDE = 2               # guidance iterations per denoising step
NORMALIZE_GRAD = True      # L2-normalize gradients per timestep
USE_ADAPTIVE = True        # cosine decay over diffusion steps
CLAMP = True               # clamp gradients
L_INF = 1.0                # clamp bound

# Sweep: (action_scale, ratio) pairs
# ratio=0 is positive-only (v0.2.2 equivalent), ratio>0 adds negative guidance
GUIDANCE_CONFIGS = [
    {"action_scale": 0.0,  "ratio": 0.0,  "label": "unguided"},
    # Positive-only baselines (v0.2.2 equivalent)
    {"action_scale": 0.1,  "ratio": 0.0,  "label": "pos_only_0.1"},
    {"action_scale": 0.2,  "ratio": 0.0,  "label": "pos_only_0.2"},
    # Full guidance: sweep ratio at action_scale=0.2 (SOPE default)
    {"action_scale": 0.2,  "ratio": 0.25, "label": "full_0.2_r0.25"},
    {"action_scale": 0.2,  "ratio": 0.5,  "label": "full_0.2_r0.5"},
    {"action_scale": 0.2,  "ratio": 0.75, "label": "full_0.2_r0.75"},
    {"action_scale": 0.2,  "ratio": 1.0,  "label": "full_0.2_r1.0"},
    # Full guidance: sweep action_scale at ratio=0.5
    {"action_scale": 0.05, "ratio": 0.5,  "label": "full_0.05_r0.5"},
    {"action_scale": 0.1,  "ratio": 0.5,  "label": "full_0.1_r0.5"},
    {"action_scale": 0.5,  "ratio": 0.5,  "label": "full_0.5_r0.5"},
    {"action_scale": 1.0,  "ratio": 0.5,  "label": "full_1.0_r0.5"},
]

print(f"state_dim={STATE_DIM}, action_dim={ACTION_DIM}, transition_dim={TRANSITION_DIM}")
print(f"k_guide={K_GUIDE}, adaptive={USE_ADAPTIVE}, clamp={CLAMP}")
print(f"Behavior policy: hidden={BEHAVIOR_HIDDEN_DIM}, steps={BEHAVIOR_DIFFUSION_STEPS}, epochs={BEHAVIOR_TRAIN_EPOCHS}")
print(f"\nGuidance configs to sweep ({len(GUIDANCE_CONFIGS)}):")
for gc in GUIDANCE_CONFIGS:
    print(f"  {gc['label']}: action_scale={gc['action_scale']}, ratio={gc['ratio']}")

state_dim=19, action_dim=7, transition_dim=26
k_guide=2, adaptive=True, clamp=True
Behavior policy: hidden=256, steps=100, epochs=1000

Guidance configs to sweep (11):
  unguided: action_scale=0.0, ratio=0.0
  pos_only_0.1: action_scale=0.1, ratio=0.0
  pos_only_0.2: action_scale=0.2, ratio=0.0
  full_0.2_r0.25: action_scale=0.2, ratio=0.25
  full_0.2_r0.5: action_scale=0.2, ratio=0.5
  full_0.2_r0.75: action_scale=0.2, ratio=0.75
  full_0.2_r1.0: action_scale=0.2, ratio=1.0
  full_0.05_r0.5: action_scale=0.05, ratio=0.5
  full_0.1_r0.5: action_scale=0.1, ratio=0.5
  full_0.5_r0.5: action_scale=0.5, ratio=0.5
  full_1.0_r0.5: action_scale=1.0, ratio=0.5


## Step 0: Load Demo Data (Full Dims)

In [3]:
def load_robomimic_demos_full(hdf5_path, obs_keys):
    """Load robomimic demos — full dims, no reduction."""
    data = []
    all_states_list = []
    all_actions_list = []
    
    with h5py.File(hdf5_path, "r") as f:
        demo_keys = sorted(f["data"].keys(), key=lambda x: int(x.split("_")[1]))
        print(f"Found {len(demo_keys)} demos")
        
        for dk in tqdm(demo_keys, desc="Loading demos"):
            demo = f["data"][dk]
            obs_arrays = [demo["obs"][k][:] for k in obs_keys]
            state = np.concatenate(obs_arrays, axis=-1).astype(np.float32)
            actions = demo["actions"][:].astype(np.float32)
            rewards = demo["rewards"][:].astype(np.float32)
            
            episode = {
                "states": state[:-1],
                "actions": actions[:-1],
                "rewards": rewards[:-1],
                "next-states": state[1:],
            }
            data.append(episode)
            all_states_list.append(state)
            all_actions_list.append(actions)
    
    all_states = np.concatenate(all_states_list, axis=0)
    all_actions = np.concatenate(all_actions_list, axis=0)
    total_transitions = sum(len(ep["states"]) for ep in data)
    print(f"Loaded {len(data)} episodes, {total_transitions} transitions")
    print(f"State shape: {data[0]['states'].shape}, Action shape: {data[0]['actions'].shape}")
    return data, all_states, all_actions

offline_data, all_states, all_actions = load_robomimic_demos_full(DEMO_HDF5, OBS_KEYS)

# ── Normalization stats ──
norm_mean = np.concatenate([all_states.mean(0), all_actions.mean(0)]).astype(np.float32)
norm_std = np.maximum(np.concatenate([all_states.std(0), all_actions.std(0)]), 1e-6).astype(np.float32)
norm_mean_t = torch.tensor(norm_mean, device=device)
norm_std_t = torch.tensor(norm_std, device=device)
normalize_fn = lambda x: (x - norm_mean_t) / norm_std_t
unnormalize_fn = lambda x: x * norm_std_t + norm_mean_t
print(f"Norm mean shape: {norm_mean.shape}")

Found 200 demos


Loading demos:   0%|          | 0/200 [00:00<?, ?it/s]

Loading demos:  32%|███▎      | 65/200 [00:00<00:00, 642.70it/s]

Loading demos:  66%|██████▌   | 131/200 [00:00<00:00, 648.71it/s]

Loading demos:  98%|█████████▊| 196/200 [00:00<00:00, 641.80it/s]

Loading demos: 100%|██████████| 200/200 [00:00<00:00, 638.82it/s]

Loaded 200 episodes, 9466 transitions
State shape: (58, 19), Action shape: (58, 7)


Norm mean shape: (26,)


## Step 1: Load Oracle Values

In [4]:
with open(ORACLE_JSON, "r") as f:
    oracle_data = json.load(f)

oracle_returns = np.array(oracle_data["returns"])
oracle_value = float(oracle_data["mean_return"])
oracle_success_rate = float(np.mean(oracle_returns > 0.5))

print(f"Oracle V^pi = {oracle_value:.4f} +/- {np.std(oracle_returns):.4f}")
print(f"Oracle success rate: {oracle_success_rate*100:.1f}%")

Oracle V^pi = 0.5400 +/- 0.4984
Oracle success rate: 54.0%


## Step 2: Load Pre-trained Chunk Diffusion Model (from v0.2.2)

In [5]:
temporal_model = TemporalUnet(
    horizon=CHUNK_SIZE,
    transition_dim=TRANSITION_DIM,
    dim=BASE_DIM,
    dim_mults=DIM_MULTS,
    attention=False,
).to(device)

diffusion_model = GaussianDiffusion(
    model=temporal_model,
    horizon=CHUNK_SIZE,
    observation_dim=STATE_DIM,
    action_dim=ACTION_DIM,
    n_timesteps=N_DIFFUSION_STEPS,
    normalizer=normalize_fn,
    unnormalizer=unnormalize_fn,
    predict_epsilon=PREDICT_EPSILON,
    loss_type="l2",
    clip_denoised=False,
    action_weight=ACTION_WEIGHT,
    loss_weights=None,
    loss_discount=1.0,
).to(device)

ema = EMA(diffusion_model)
ema_ckpt = torch.load(DIFFUSION_SAVE_DIR / "diffusion_model_ema.pt", map_location=device)
ema.ema_model.load_state_dict(ema_ckpt)
model_ckpt = torch.load(DIFFUSION_SAVE_DIR / "diffusion_model.pt", map_location=device)
diffusion_model.load_state_dict(model_ckpt)

n_params = sum(p.numel() for p in diffusion_model.parameters())
print(f"Loaded chunk diffuser from {DIFFUSION_SAVE_DIR}")
print(f"Parameters: {n_params:,}")

[ models/temporal ] Channel dimensions: [(26, 32), (32, 128), (128, 256)]
[(26, 32), (32, 128), (128, 256)]


Loaded chunk diffuser from /home1/reishuen/latent_sope/diffusion_ckpts/mvp_v022_fulldim
Parameters: 3,686,618


/home1/reishuen/latent_sope/third_party/sope/opelab/core/baselines/diffusion/diffusion.py:314: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  betas * np.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod))


## Step 3: Train Diffusion Behavior Policy on Expert Demos

Train a simple MLP-based DDPM that models β(a|s) on expert demo (state, action) pairs.
This is a single-step action diffusion model (NOT a trajectory chunk diffuser).

Score extraction: evaluate noise prediction at t=1, return `-noise_pred / sigma[1]`.
This is a novel choice — SOPE uses BC_Gaussian/TD3 for behavior policies, not diffusion.
But the score function gives the same ∇_a log β(a|s) gradient.

In [6]:
# ── Diffusion Behavior Policy: MLP-based DDPM over actions conditioned on states ──

class SinusoidalPosEmb(nn.Module):
    """Sinusoidal positional embedding for diffusion timesteps."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device, dtype=torch.float32) * -emb)
        emb = t.float().unsqueeze(-1) * emb.unsqueeze(0)
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class ActionDenoiser(nn.Module):
    """MLP denoiser: predicts noise given (noisy_action, timestep, state).
    
    Architecture mirrors SOPE's PearceMlp: embed action + timestep + state,
    then process through residual FC blocks.
    """
    def __init__(self, state_dim, action_dim, hidden_dim=256, time_emb_dim=64):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalPosEmb(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.Mish(),
        )
        input_dim = action_dim + time_emb_dim + state_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Mish(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Mish(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Mish(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, noisy_action, timestep, state):
        """
        Args:
            noisy_action: (B, action_dim)
            timestep: (B,) integer timesteps
            state: (B, state_dim)
        Returns:
            noise_pred: (B, action_dim)
        """
        t_emb = self.time_embed(timestep)
        x = torch.cat([noisy_action, t_emb, state], dim=-1)
        return self.net(x)


class DiffusionBehaviorPolicy:
    """Single-step DDPM over actions conditioned on states.
    
    Trains: p(a|s) via denoising diffusion on expert demo (state, action) pairs.
    Score:  grad_log_prob(s, a) = -noise_pred(a, t=1, s) / sigma[1]
    
    Uses cosine beta schedule matching SOPE's convention.
    """
    def __init__(self, state_dim, action_dim, hidden_dim=256, n_timesteps=100,
                 device="cuda"):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.n_timesteps = n_timesteps
        self.device = torch.device(device)

        # Denoiser network
        self.model = ActionDenoiser(
            state_dim=state_dim,
            action_dim=action_dim,
            hidden_dim=hidden_dim,
        ).to(self.device)

        # Cosine beta schedule (matching SOPE)
        steps = n_timesteps + 1
        t = torch.linspace(0, n_timesteps, steps) / n_timesteps
        alphas_cumprod = torch.cos((t + 0.008) / 1.008 * math.pi / 2) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        betas = betas.clamp(max=0.999)

        alphas = 1.0 - betas
        self.alphas_cumprod = torch.cumprod(alphas, dim=0).to(self.device)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod).to(self.device)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod).to(self.device)

        # sigma[t] = sqrt(1 - alpha_bar[t])
        self.sigma = self.sqrt_one_minus_alphas_cumprod

        # Score timestep (t=1, near-clean)
        self.score_timestep = 1
        self.score_sigma = self.sigma[self.score_timestep].item()
        
        print(f"DiffusionBehaviorPolicy: {sum(p.numel() for p in self.model.parameters()):,} params")
        print(f"  n_timesteps={n_timesteps}, sigma[1]={self.score_sigma:.6f}")

    def train(self, states, actions, epochs=200, batch_size=256, lr=1e-3):
        """Train the denoiser on (state, action) pairs."""
        self.model.train()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        
        states_t = torch.tensor(states, dtype=torch.float32, device=self.device)
        actions_t = torch.tensor(actions, dtype=torch.float32, device=self.device)
        n_samples = len(states_t)
        
        losses = []
        for epoch in range(epochs):
            # Random minibatch
            idx = torch.randint(0, n_samples, (batch_size,), device=self.device)
            s_batch = states_t[idx]
            a_batch = actions_t[idx]
            
            # Sample random timesteps
            t = torch.randint(0, self.n_timesteps, (batch_size,), device=self.device)
            
            # Forward diffusion: add noise to actions
            noise = torch.randn_like(a_batch)
            sqrt_alpha = self.sqrt_alphas_cumprod[t].unsqueeze(-1)
            sqrt_one_minus_alpha = self.sqrt_one_minus_alphas_cumprod[t].unsqueeze(-1)
            noisy_actions = sqrt_alpha * a_batch + sqrt_one_minus_alpha * noise
            
            # Predict noise
            noise_pred = self.model(noisy_actions, t, s_batch)
            
            # MSE loss
            loss = nn.functional.mse_loss(noise_pred, noise)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            
            if (epoch + 1) % 100 == 0:
                print(f"  Epoch {epoch+1}/{epochs}, loss={loss.item():.6f}")
        
        self.model.eval()
        return losses

    @torch.no_grad()
    def grad_log_prob(self, states, actions):
        """Compute score: nabla_a log beta(a|s) = -noise_pred(a, t=1, s) / sigma[1].
        
        Args:
            states: (N, state_dim)
            actions: (N, action_dim)
        Returns:
            score: (N, action_dim)
        """
        states = states.to(self.device)
        actions = actions.to(self.device)
        t = torch.full((states.shape[0],), self.score_timestep,
                       device=self.device, dtype=torch.long)
        noise_pred = self.model(actions, t, states)
        return -noise_pred / self.score_sigma

    @torch.no_grad()
    def grad_log_prob_chunk(self, states, actions):
        """Chunk-level scoring: flatten (B, T, D) to (B*T, D), score, reshape back.
        
        Args:
            states: (B, T, state_dim)
            actions: (B, T, action_dim)
        Returns:
            score: (B, T, action_dim)
        """
        B, T, _ = states.shape
        s_flat = states.reshape(B * T, -1)
        a_flat = actions.reshape(B * T, -1)
        score_flat = self.grad_log_prob(s_flat, a_flat)
        return score_flat.reshape(B, T, self.action_dim)


print("DiffusionBehaviorPolicy defined.")

DiffusionBehaviorPolicy defined.


In [7]:
# ── Collect expert demo (state, action) pairs and train behavior policy ──
demo_states = np.concatenate([ep["states"] for ep in offline_data], axis=0)  # (N, 19)
demo_actions = np.concatenate([ep["actions"] for ep in offline_data], axis=0)  # (N, 7)
print(f"Expert demo pairs: {demo_states.shape[0]} (states: {demo_states.shape}, actions: {demo_actions.shape})")

behavior_policy = DiffusionBehaviorPolicy(
    state_dim=STATE_DIM,
    action_dim=ACTION_DIM,
    hidden_dim=BEHAVIOR_HIDDEN_DIM,
    n_timesteps=BEHAVIOR_DIFFUSION_STEPS,
    device=str(device),
)

print(f"\nTraining behavior policy on {demo_states.shape[0]} expert demo pairs...")
behavior_losses = behavior_policy.train(
    demo_states, demo_actions,
    epochs=BEHAVIOR_TRAIN_EPOCHS,
    batch_size=BC_BATCH_SIZE,
    lr=BC_LR,
)
print(f"Final loss: {behavior_losses[-1]:.6f}")

# Sanity check: score a few demo actions
test_s = torch.tensor(demo_states[:8], device=device)
test_a = torch.tensor(demo_actions[:8], device=device)
test_score = behavior_policy.grad_log_prob(test_s, test_a)
print(f"\nSanity check — score norms on demo data: {test_score.norm(dim=-1).cpu().numpy()}")
print(f"Score mean magnitude: {test_score.abs().mean().item():.2f}")

Expert demo pairs: 9466 (states: (9466, 19), actions: (9466, 7))
DiffusionBehaviorPolicy: 160,839 params
  n_timesteps=100, sigma[1]=0.041803

Training behavior policy on 9466 expert demo pairs...


  Epoch 100/1000, loss=0.195552


  Epoch 200/1000, loss=0.154492


  Epoch 300/1000, loss=0.138848


  Epoch 400/1000, loss=0.132112


...

  Epoch 600/1000, loss=0.126905


  Epoch 700/1000, loss=0.096149


  Epoch 800/1000, loss=0.122159


  Epoch 900/1000, loss=0.124319


  Epoch 1000/1000, loss=0.097694
Final loss: 0.097694

Sanity check — score norms on demo data: [38.02565  25.663029 21.3219   20.601788 29.05733  28.680567 30.224485
 38.800003]
Score mean magnitude: 8.54


## Step 4: Load Target Policy Scorer

Same as v0.2.2 — `RobomimicDiffusionScorer` wraps the target policy to provide
`grad_log_prob_chunk()` via the diffusion score function at t=1.

In [8]:
ckpt = load_checkpoint(CKPT_DIR, ckpt_path=Path("last.pth"))
target_algo = build_algo_from_checkpoint(ckpt, device=str(device))

target_scorer = RobomimicDiffusionScorer(target_algo, device=str(device), obs_keys=OBS_KEYS)
print(f"Target scorer: state_dim={target_scorer.state_dim}, action_dim={target_scorer.action_dim}")
print(f"  sigma[1]={target_scorer.sigma:.6f}, prediction_horizon={target_scorer.prediction_horizon}")

# Quick sanity: compare target vs behavior scores on same data
test_s = torch.tensor(demo_states[:8], device=device)
test_a = torch.tensor(demo_actions[:8], device=device)
target_score = target_scorer.grad_log_prob(test_s, test_a)
behavior_score = behavior_policy.grad_log_prob(test_s, test_a)
print(f"\nTarget score norms:   {target_score.norm(dim=-1).cpu().numpy()}")
print(f"Behavior score norms: {behavior_score.norm(dim=-1).cpu().numpy()}")
# If these are very different in magnitude, normalization is important
print(f"Target/Behavior norm ratio: {(target_score.norm(dim=-1) / (behavior_score.norm(dim=-1) + 1e-8)).mean().item():.2f}")


============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_gripper_qpos', 'object', 'robot0_eef_quat', 'robot0_eef_pos']
using obs modality: rgb with keys: []
using obs modality: depth with keys: []
using obs modality: scan with keys: []


number of parameters: 6.576359e+07


[09:35:02] INFO     build_algo_from_checkpoint took 0.47 seconds to execute                           ]8;id=362688;file:///home1/reishuen/latent_sope/src/latent_sope/utils/common.py\common.py]8;;\:]8;id=53580;file:///home1/reishuen/latent_sope/src/latent_sope/utils/common.py#137\137]8;;\

Target scorer: state_dim=19, action_dim=7
  sigma[1]=0.041803, prediction_horizon=16



Target score norms:   [15.22282  15.149933 16.024246 15.994344 15.700259 14.682028 14.490031
 16.132196]
Behavior score norms: [38.02565  25.663029 21.3219   20.601788 29.05733  28.680567 30.224485
 38.800003]
Target/Behavior norm ratio: 0.56


## Step 5: Guided Stitching with Full SOPE Guidance

The generation function now matches SOPE's `default_sample_fn()`:
1. `k_guide` iterations per denoising step
2. Target gradient (diffusion score) − `ratio` × behavior gradient (diffusion score)
3. Per-timestep L2 normalization + clamping
4. Adaptive cosine scaling over diffusion timesteps
5. Re-normalize → re-condition → unnormalize after each guidance update

In [9]:
def get_schedule_multiplier(t, n_timesteps):
    """Cosine schedule multiplier for adaptive guidance scaling.
    Matches SOPE's get_schedule_multiplier() in diffusion.py:112-118.
    """
    t_frac = t / n_timesteps
    return 0.5 * (1 + math.cos(math.pi * t_frac))


def get_initial_states_from_data(offline_data, num_samples, device):
    """Sample initial states from the demo dataset."""
    all_initial = np.array([ep["states"][0] for ep in offline_data])
    indices = np.random.choice(len(all_initial), num_samples, replace=True)
    return torch.tensor(all_initial[indices], dtype=torch.float32, device=device)


def generate_trajectories_full_guidance(
    diffusion_model, initial_states,
    normalize_fn, unnormalize_fn,
    state_dim, action_dim,
    chunk_size, t_gen, device,
    target_scorer=None, behavior_scorer=None,
    action_scale=0.0, ratio=0.5,
    k_guide=2, normalize_grad=True,
    use_adaptive=True, clamp=True, l_inf=1.0,
):
    """Generate trajectories with full SOPE-style guidance (positive + negative).
    
    Matches SOPE's default_sample_fn() from diffusion.py:152-250:
    - target_scorer provides nabla_a log pi(a|s)  (positive term)
    - behavior_scorer provides nabla_a log beta(a|s)  (negative term)
    - guide = target_grad - ratio * behavior_grad
    - k_guide iterations per denoising step
    - Adaptive cosine scaling over diffusion timesteps
    """
    guided = (target_scorer is not None and action_scale > 0)
    use_neg_grad = (behavior_scorer is not None and ratio > 0)
    batch_size = initial_states.shape[0]
    transition_dim = state_dim + action_dim
    n_timesteps = diffusion_model.n_timesteps

    # Normalize initial states for conditioning
    padded = torch.cat([
        initial_states,
        torch.zeros(batch_size, action_dim, device=device)
    ], dim=1)
    normalized_initial = normalize_fn(padded)[:, :state_dim]

    all_trajectories = torch.zeros(batch_size, t_gen, transition_dim, device=device)
    conditions = {0: normalized_initial}
    total_generated = 0
    n_iterations = 0

    while total_generated < t_gen:
        n_iterations += 1
        steps_remaining = t_gen - total_generated
        shape = (batch_size, chunk_size, transition_dim)

        # Initialize from noise
        x = torch.randn(shape, device=device)
        x = apply_conditioning(x, conditions, state_dim)

        for t_diff in reversed(range(n_timesteps)):
            t_tensor = torch.full((batch_size,), t_diff, device=device, dtype=torch.long)

            with torch.no_grad():
                model_mean, _, model_log_variance = diffusion_model.p_mean_variance(x=x, t=t_tensor)
                model_std = torch.exp(0.5 * model_log_variance)

            if guided:
                # Unnormalize to real space (guidance operates in original data space)
                model_mean = unnormalize_fn(model_mean)

                for _k in range(k_guide):
                    # Extract states and actions from current mean
                    states_chunk = model_mean[:, :, :state_dim]
                    actions_chunk = model_mean[:, :, state_dim:]

                    # Positive: target policy score
                    target_grad = target_scorer.grad_log_prob_chunk(states_chunk, actions_chunk)
                    # target_grad: (B, T, action_dim)

                    # Negative: behavior policy score
                    if use_neg_grad:
                        behavior_grad = behavior_scorer.grad_log_prob_chunk(states_chunk, actions_chunk)

                    # Per-timestep L2 normalization (SOPE default)
                    if normalize_grad:
                        eps = 1e-6
                        target_norm = target_grad.norm(dim=-1, keepdim=True) + eps
                        target_grad = target_grad / target_norm
                        if use_neg_grad:
                            behavior_norm = behavior_grad.norm(dim=-1, keepdim=True) + eps
                            behavior_grad = behavior_grad / behavior_norm

                    # Clamp (SOPE default)
                    if clamp:
                        target_grad = target_grad.clamp(-l_inf, l_inf)
                        if use_neg_grad:
                            behavior_grad = (ratio * behavior_grad).clamp(-l_inf, l_inf)

                    # Combine: guide = target - ratio * behavior
                    if use_neg_grad:
                        if clamp:
                            guide_actions = target_grad - behavior_grad  # already scaled by ratio
                        else:
                            guide_actions = target_grad - ratio * behavior_grad
                    else:
                        guide_actions = target_grad

                    # Build full guide tensor (zeros for states)
                    guide = torch.zeros_like(model_mean)
                    guide[:, :, state_dim:] = guide_actions

                    # Adaptive cosine scaling (SOPE default)
                    if use_adaptive:
                        scale_mult = get_schedule_multiplier(n_timesteps - t_diff, n_timesteps)
                        guide = scale_mult * action_scale * guide
                    else:
                        guide = action_scale * guide

                    # Apply guidance
                    model_mean = model_mean + guide

                    # Re-normalize, re-condition, un-normalize
                    model_mean = normalize_fn(model_mean)
                    model_mean = apply_conditioning(model_mean, conditions, state_dim)
                    model_mean = unnormalize_fn(model_mean)

                # Final re-normalize before adding noise
                model_mean = normalize_fn(model_mean)

            # Add noise (standard diffusion sampling)
            noise = torch.randn_like(x)
            nonzero_mask = (1 - (t_diff == 0) * 1.0)
            x = model_mean + nonzero_mask * model_std * noise
            x = apply_conditioning(x, conditions, state_dim)

        # Unnormalize the generated chunk
        chunk_unnorm = unnormalize_fn(x)

        # Store chunk (1-step overlap)
        steps_to_store = min(chunk_size - 1, steps_remaining)
        all_trajectories[:, total_generated:total_generated + steps_to_store] = chunk_unnorm[:, :steps_to_store]
        total_generated += steps_to_store

        if total_generated >= t_gen:
            break

        # Condition next chunk on last state (stay in normalized space)
        last_states_norm = x[:, -1, :state_dim]
        conditions = {0: last_states_norm}

    label = "unguided"
    if guided and use_neg_grad:
        label = f"full(s={action_scale},r={ratio})"
    elif guided:
        label = f"pos_only(s={action_scale})"
    print(f"  [{label}] Generated {total_generated} steps in {n_iterations} chunks")
    return all_trajectories.detach().cpu().numpy()


def score_trajectories_gt(trajectories, cube_z_index, threshold, gamma=1.0):
    """Score trajectories using ground-truth Lift reward."""
    B, T, D = trajectories.shape
    returns = np.zeros(B)
    successes = np.zeros(B, dtype=bool)
    for i in range(B):
        gamma_t = 1.0
        for t in range(T):
            reward = 1.0 if trajectories[i, t, cube_z_index] > threshold else 0.0
            returns[i] += reward * gamma_t
            gamma_t *= gamma
            if reward > 0:
                successes[i] = True
    return returns, successes


print("Generation and scoring functions defined.")

Generation and scoring functions defined.


## Step 6: Run Guidance Sweep

In [10]:
np.random.seed(42)
torch.manual_seed(42)

initial_states = get_initial_states_from_data(offline_data, NUM_SYNTHETIC_TRAJS, device)
results_by_config = {}

for i, gc in enumerate(GUIDANCE_CONFIGS):
    label = gc["label"]
    action_scale = gc["action_scale"]
    ratio = gc["ratio"]

    print("=" * 60)
    print(f"[{i+1}/{len(GUIDANCE_CONFIGS)}] {label}")
    print("=" * 60)

    t0 = time.time()

    trajs = generate_trajectories_full_guidance(
        diffusion_model=ema.ema_model,
        initial_states=initial_states,
        normalize_fn=normalize_fn, unnormalize_fn=unnormalize_fn,
        state_dim=STATE_DIM, action_dim=ACTION_DIM,
        chunk_size=CHUNK_SIZE, t_gen=T_GEN, device=device,
        target_scorer=target_scorer if action_scale > 0 else None,
        behavior_scorer=behavior_policy if (action_scale > 0 and ratio > 0) else None,
        action_scale=action_scale, ratio=ratio,
        k_guide=K_GUIDE, normalize_grad=NORMALIZE_GRAD,
        use_adaptive=USE_ADAPTIVE, clamp=CLAMP, l_inf=L_INF,
    )

    elapsed = time.time() - t0
    returns, successes = score_trajectories_gt(trajs, CUBE_Z_INDEX, LIFT_THRESHOLD, GAMMA)

    rel_err = abs(np.mean(returns) - oracle_value) / abs(oracle_value) if oracle_value != 0 else float('inf')

    results_by_config[label] = {
        "trajs": trajs,
        "returns": returns,
        "successes": successes,
        "estimate": float(np.mean(returns)),
        "std": float(np.std(returns)),
        "success_rate": float(np.mean(successes)),
        "rel_error": rel_err,
        "action_scale": action_scale,
        "ratio": ratio,
        "time": elapsed,
    }
    print(f"  OPE={np.mean(returns):.4f}, SR={np.mean(successes)*100:.1f}%, "
          f"rel_err={rel_err:.2%}, time={elapsed:.0f}s\n")

# Summary table
print("=" * 70)
print("GUIDANCE SWEEP SUMMARY — v0.2.3 (Positive + Negative Guidance)")
print("=" * 70)
print(f"Oracle V^pi = {oracle_value:.4f} (SR = {oracle_success_rate*100:.1f}%)")
print(f"\n{'Config':<22} {'Scale':>6} {'Ratio':>6} {'OPE':>8} {'SR':>6} {'RelErr':>10}")
print("-" * 62)
for label, r in results_by_config.items():
    print(f"{label:<22} {r['action_scale']:>6.2f} {r['ratio']:>6.2f} "
          f"{r['estimate']:>8.4f} {r['success_rate']*100:>5.1f}% {r['rel_error']:>9.2%}")

[1/11] unguided


  [unguided] Generated 60 steps in 20 chunks
  OPE=15.7800, SR=98.0%, rel_err=2822.22%, time=49s

[2/11] pos_only_0.1


  [pos_only(s=0.1)] Generated 60 steps in 20 chunks
  OPE=15.6000, SR=98.0%, rel_err=2788.89%, time=219s

[3/11] pos_only_0.2


  [pos_only(s=0.2)] Generated 60 steps in 20 chunks
  OPE=15.5000, SR=98.0%, rel_err=2770.37%, time=219s

[4/11] full_0.2_r0.25


  [full(s=0.2,r=0.25)] Generated 60 steps in 20 chunks
  OPE=15.7000, SR=98.0%, rel_err=2807.41%, time=221s

[5/11] full_0.2_r0.5


...

  [full(s=0.2,r=1.0)] Generated 60 steps in 20 chunks
  OPE=15.9600, SR=98.0%, rel_err=2855.56%, time=221s

[8/11] full_0.05_r0.5


  [full(s=0.05,r=0.5)] Generated 60 steps in 20 chunks
  OPE=15.9800, SR=98.0%, rel_err=2859.26%, time=221s

[9/11] full_0.1_r0.5


  [full(s=0.1,r=0.5)] Generated 60 steps in 20 chunks
  OPE=15.7400, SR=98.0%, rel_err=2814.81%, time=221s

[10/11] full_0.5_r0.5


  [full(s=0.5,r=0.5)] Generated 60 steps in 20 chunks
  OPE=15.9600, SR=98.0%, rel_err=2855.56%, time=221s

[11/11] full_1.0_r0.5


  [full(s=1.0,r=0.5)] Generated 60 steps in 20 chunks
  OPE=16.0800, SR=98.0%, rel_err=2877.78%, time=221s

GUIDANCE SWEEP SUMMARY — v0.2.3 (Positive + Negative Guidance)
Oracle V^pi = 0.5400 (SR = 54.0%)

Config                  Scale  Ratio      OPE     SR     RelErr
--------------------------------------------------------------
unguided                 0.00   0.00  15.7800  98.0%  2822.22%
pos_only_0.1             0.10   0.00  15.6000  98.0%  2788.89%
pos_only_0.2             0.20   0.00  15.5000  98.0%  2770.37%
full_0.2_r0.25           0.20   0.25  15.7000  98.0%  2807.41%
full_0.2_r0.5            0.20   0.50  15.8200  98.0%  2829.63%
full_0.2_r0.75           0.20   0.75  16.0200  98.0%  2866.67%
full_0.2_r1.0            0.20   1.00  15.9600  98.0%  2855.56%
full_0.05_r0.5           0.05   0.50  15.9800  98.0%  2859.26%
full_0.1_r0.5            0.10   0.50  15.7400  98.0%  2814.81%
full_0.5_r0.5            0.50   0.50  15.9600  98.0%  2855.56%
full_1.0_r0.5            1.00   0.50 

## Step 7: Visualization

In [11]:
# ── Find best config ──
best_label = min(results_by_config, key=lambda k: results_by_config[k]["rel_error"])
best_r = results_by_config[best_label]
print(f"Best config: {best_label} (rel_error={best_r['rel_error']:.2%})")

# ── Figure 1: OPE + Success Rate summary ──
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

labels = list(results_by_config.keys())
estimates = [results_by_config[l]["estimate"] for l in labels]
stds = [results_by_config[l]["std"] for l in labels]
success_rates = [results_by_config[l]["success_rate"] * 100 for l in labels]

# Color: blue=unguided, orange=pos_only, red=full guidance
colors = []
for l in labels:
    if "unguided" in l:
        colors.append("steelblue")
    elif "pos_only" in l:
        colors.append("orange")
    else:
        colors.append("coral")

# Panel 1: OPE estimates
axes[0].bar(range(len(labels)), estimates, color=colors, edgecolor="black")
axes[0].errorbar(range(len(labels)), estimates, yerr=stds, fmt="none", color="black", capsize=3)
axes[0].axhline(y=oracle_value, color="green", linestyle="--", linewidth=2, label=f"Oracle={oracle_value:.2f}")
axes[0].set_xticks(range(len(labels)))
axes[0].set_xticklabels(labels, rotation=60, ha="right", fontsize=7)
axes[0].set_ylabel("OPE Estimate")
axes[0].set_title("OPE by Guidance Config")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3, axis="y")

# Panel 2: Success rates
axes[1].bar(range(len(labels)), success_rates, color=colors, edgecolor="black")
axes[1].axhline(y=oracle_success_rate * 100, color="green", linestyle="--", linewidth=2, label=f"Oracle={oracle_success_rate*100:.0f}%")
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels, rotation=60, ha="right", fontsize=7)
axes[1].set_ylabel("Success Rate (%)")
axes[1].set_title("Synthetic Success Rate")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis="y")

# Panel 3: Cube z for best config vs unguided
unguided_trajs = results_by_config["unguided"]["trajs"]
best_trajs = best_r["trajs"]
for j in range(min(10, NUM_SYNTHETIC_TRAJS)):
    axes[2].plot(unguided_trajs[j, :, CUBE_Z_INDEX], alpha=0.2, color="steelblue",
                 label="Unguided" if j == 0 else "")
    axes[2].plot(best_trajs[j, :, CUBE_Z_INDEX], alpha=0.2, color="coral",
                 label=f"Best ({best_label})" if j == 0 else "")
axes[2].axhline(y=LIFT_THRESHOLD, color="red", linestyle=":", alpha=0.5, label="Lift threshold")
axes[2].set_xlabel("Timestep")
axes[2].set_ylabel("Cube z")
axes[2].set_title("Trajectory Quality")
axes[2].legend(fontsize=7)
axes[2].grid(True, alpha=0.3)

plt.suptitle("MVP v0.2.3: Full SOPE Guidance (Positive + Negative)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ope_summary_mvp_v023.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'ope_summary_mvp_v023.png'}")

Best config: pos_only_0.2 (rel_error=2770.37%)


Saved: /home1/reishuen/latent_sope/results/2026-03-12/ope_summary_mvp_v023.png


In [12]:
# ── Figure 2: Per-dimension trajectory comparison (best vs unguided vs demos) ──
KEY_DIMS = {
    "cube_z": 2, "cube_x": 0, "eef_z": 12, "eef_x": 10,
    "grip2cube_z": 9, "gripper_q0": 17,
}

# Real demo trajectories for comparison
demo_trajs = []
for ep in offline_data[:20]:
    traj = np.concatenate([ep["states"], ep["actions"]], axis=-1)
    if len(traj) >= T_GEN:
        demo_trajs.append(traj[:T_GEN])
    else:
        padded = np.zeros((T_GEN, TRANSITION_DIM), dtype=np.float32)
        padded[:len(traj)] = traj
        padded[len(traj):] = traj[-1]
        demo_trajs.append(padded)
demo_trajs = np.array(demo_trajs)

N_SHOW = 10
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, dim) in enumerate(KEY_DIMS.items()):
    ax = axes[idx]
    for j in range(min(N_SHOW, len(demo_trajs))):
        ax.plot(demo_trajs[j, :, dim], color="green", alpha=0.15,
                label="Demo" if j == 0 else "")
    for j in range(min(N_SHOW, NUM_SYNTHETIC_TRAJS)):
        ax.plot(unguided_trajs[j, :, dim], color="steelblue", alpha=0.25,
                label="Unguided" if j == 0 else "")
    for j in range(min(N_SHOW, NUM_SYNTHETIC_TRAJS)):
        ax.plot(best_trajs[j, :, dim], color="coral", alpha=0.25,
                label=f"Best ({best_label})" if j == 0 else "")
    if dim == CUBE_Z_INDEX:
        ax.axhline(y=LIFT_THRESHOLD, color="red", linestyle=":", alpha=0.5)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Timestep")
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=7, loc="upper left")

plt.suptitle("State Trajectories: Demo (green) vs Unguided (blue) vs Best Guided (red)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "traj_states_v023.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'traj_states_v023.png'}")

Saved: /home1/reishuen/latent_sope/results/2026-03-12/traj_states_v023.png


In [13]:
# ── Figure 3: Cube z across ALL configs ──
n_configs = len(results_by_config)
n_cols = 4
n_rows = math.ceil(n_configs / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for idx, (label, r) in enumerate(results_by_config.items()):
    ax = axes[idx]
    trajs = r["trajs"]
    for j in range(min(5, len(demo_trajs))):
        ax.plot(demo_trajs[j, :, CUBE_Z_INDEX], color="green", alpha=0.1)
    for j in range(min(15, NUM_SYNTHETIC_TRAJS)):
        ax.plot(trajs[j, :, CUBE_Z_INDEX], color="coral", alpha=0.2)
    ax.plot(trajs[:, :, CUBE_Z_INDEX].mean(axis=0), color="darkred", linewidth=2, label="Mean")
    ax.plot(demo_trajs[:, :, CUBE_Z_INDEX].mean(axis=0), color="darkgreen", linewidth=2,
            linestyle="--", label="Demo mean")
    ax.axhline(y=LIFT_THRESHOLD, color="red", linestyle=":", alpha=0.5)
    ax.set_title(f"{label}\nOPE={r['estimate']:.2f}, SR={r['success_rate']*100:.0f}%", fontsize=9)
    ax.set_xlabel("Timestep")
    ax.set_ylabel("cube_z")
    ax.set_ylim([0.78, 0.95])
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=7)

for idx in range(n_configs, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle("Cube Z Across All Guidance Configs", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "traj_cubez_all_v023.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'traj_cubez_all_v023.png'}")

Saved: /home1/reishuen/latent_sope/results/2026-03-12/traj_cubez_all_v023.png


In [14]:
# ── Figure 4: Behavior policy training loss ──
fig, ax = plt.subplots(1, 1, figsize=(8, 3))
ax.plot(behavior_losses)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title(f"Diffusion Behavior Policy Training ({BEHAVIOR_TRAIN_EPOCHS} epochs)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "behavior_policy_loss_v023.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {RESULTS_DIR / 'behavior_policy_loss_v023.png'}")

Saved: /home1/reishuen/latent_sope/results/2026-03-12/behavior_policy_loss_v023.png


## Step 8: Save Results

In [15]:
# ── Save results JSON ──
results_json = {
    "experiment": "MVP_v0.2.3_negative_guidance",
    "date": "2026-03-12",
    "config": {
        "state_dim": STATE_DIM,
        "action_dim": ACTION_DIM,
        "transition_dim": TRANSITION_DIM,
        "chunk_size": CHUNK_SIZE,
        "n_diffusion_steps": N_DIFFUSION_STEPS,
        "dim_mults": list(DIM_MULTS),
        "predict_epsilon": PREDICT_EPSILON,
        "num_synthetic_trajs": NUM_SYNTHETIC_TRAJS,
        "t_gen": T_GEN,
        "guidance": {
            "k_guide": K_GUIDE,
            "normalize_grad": NORMALIZE_GRAD,
            "use_adaptive": USE_ADAPTIVE,
            "clamp": CLAMP,
            "l_inf": L_INF,
        },
        "behavior_policy": {
            "type": "DiffusionBehaviorPolicy",
            "hidden_dim": BEHAVIOR_HIDDEN_DIM,
            "n_timesteps": BEHAVIOR_DIFFUSION_STEPS,
            "train_epochs": BEHAVIOR_TRAIN_EPOCHS,
            "score_sigma": behavior_policy.score_sigma,
            "final_loss": behavior_losses[-1],
            "n_params": sum(p.numel() for p in behavior_policy.model.parameters()),
        },
        "target_scorer": {
            "type": "RobomimicDiffusionScorer",
            "score_sigma": target_scorer.sigma,
            "prediction_horizon": target_scorer.prediction_horizon,
        },
    },
    "oracle": {
        "value": oracle_value,
        "success_rate": oracle_success_rate,
        "std": float(np.std(oracle_returns)),
    },
    "results_by_config": {
        label: {
            "action_scale": r["action_scale"],
            "ratio": r["ratio"],
            "estimate": r["estimate"],
            "std": r["std"],
            "success_rate": r["success_rate"],
            "rel_error": r["rel_error"],
            "time": r["time"],
            "returns": r["returns"].tolist(),
        }
        for label, r in results_by_config.items()
    },
    "best_config": best_label,
    "best_rel_error": best_r["rel_error"],
}

results_path = RESULTS_DIR / "mvp_v023_results.json"
with open(results_path, "w") as f:
    json.dump(results_json, f, indent=2)

print(f"Results saved: {results_path}")
print(f"\n{'='*60}")
print("MVP v0.2.3 COMPLETE")
print(f"{'='*60}")
print(f"Best: {best_label}, OPE={best_r['estimate']:.4f}, rel_error={best_r['rel_error']:.2%}")
print(f"Oracle: V^pi={oracle_value:.4f}")
print(f"\nComparison:")
print(f"  v0.2   (BC_Gaussian proxy, pos only, 15-dim): best rel_error=25.93%")
print(f"  v0.2.2 (diffusion score, pos only, 26-dim):   best rel_error=2533.33%")
print(f"  v0.2.3 (diffusion score, pos+neg, 26-dim):    best rel_error={best_r['rel_error']:.2%}")

Results saved: /home1/reishuen/latent_sope/results/2026-03-12/mvp_v023_results.json

MVP v0.2.3 COMPLETE
Best: pos_only_0.2, OPE=15.5000, rel_error=2770.37%
Oracle: V^pi=0.5400

Comparison:
  v0.2   (BC_Gaussian proxy, pos only, 15-dim): best rel_error=25.93%
  v0.2.2 (diffusion score, pos only, 26-dim):   best rel_error=2533.33%
  v0.2.3 (diffusion score, pos+neg, 26-dim):    best rel_error=2770.37%
